In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Load Data
df = pd.read_csv("churn_data.csv")
X = df.drop("Churn", axis=1)
y = df["Churn"].apply(lambda x: 1 if x == "Yes" else 0)

# 2. Define Feature Groups
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = ["gender", "InternetService", "Contract"]

# 3. Create Preprocessing Pipeline
# Scaler for numbers, OneHot for categories
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# 4. Create the Base Pipeline
clf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression()) # Placeholder
])

# 5. GridSearchCV for Hyperparameter Tuning
# We test two different models and their settings
param_grid = [
    {
        'classifier': [LogisticRegression(max_iter=1000)],
        'classifier__C': [0.1, 1, 10]
    },
    {
        'classifier': [RandomForestClassifier()],
        'classifier__n_estimators': [50, 100],
        'classifier__max_depth': [None, 5, 10]
    }
]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

grid_search = GridSearchCV(clf_pipeline, param_grid, cv=5, scoring='accuracy', verbose=1)
grid_search.fit(X_train, y_train)

# 6. Evaluation
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"Best Params: {grid_search.best_params_}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred))

# 7. Export the complete Pipeline
joblib.dump(best_model, "churn_pipeline.joblib")
print("✅ Pipeline exported as churn_pipeline.joblib")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best Params: {'classifier': RandomForestClassifier(), 'classifier__max_depth': None, 'classifier__n_estimators': 50}
Accuracy: 99.00%
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       198
           1       0.50      1.00      0.67         2

    accuracy                           0.99       200
   macro avg       0.75      0.99      0.83       200
weighted avg       0.99      0.99      0.99       200

✅ Pipeline exported as churn_pipeline.joblib
